In [1]:
import pandas as pd

X_train = pd.read_csv("C:/Users/AnthonySAINDOY-EODev/Downloads/X_train_update.csv")
y_train = pd.read_csv("C:/Users/AnthonySAINDOY-EODev/Downloads/Y_train_CVw08PX.csv")

df = X_train.merge(y_train, left_index=True, right_on="Unnamed: 0")
df.head()

,Unnamed: 0,Unnamed: 0_x,designation,description,productid,imageid,Unnamed: 0_y,prdtypecode
0,0,0,Olivia: Personalisiertes Notizbuch / 150 Seite...,NaN,3804725264,1263597046,0,10
1,1,1,Journal Des Arts (Le) N° 133 Du 28/09/2001 - L...,NaN,436067568,1008141237,1,2280
2,2,2,Grand Stylet Ergonomique Bleu Gamepad Nintendo...,PILOT STYLE Touch Pen de marque Speedlink est ...,201115110,938777978,2,50
3,3,3,Peluche Donald - Europe - Disneyland 2000 (Mar...,NaN,50418756,457047496,3,1280
4,4,4,La Guerre Des Tuques,Luc a des id&eacute;es de grandeur. Il veut or...,278535884,1077757786,4,2705


In [ ]:
import re
from bs4 import BeautifulSoup

def clean_text(text):
    
    if pd.isna(text):
        return ""
    
    text = BeautifulSoup(text, "html.parser").get_text()
    text = text.lower()
    text = re.sub(r"[^a-zA-Zàâäéèêëîïôöùûüç0-9 ]", " ", text)
    text = re.sub(r"\s+", " ", text)
    
    return text.strip()


df["text"] = df["designation"].fillna("") + " " + df["description"].fillna("")
df["text_clean"] = df["text"].apply(clean_text)

C:\Users\AnthonySAINDOY-EODev\AppData\Local\Temp\ipykernel_1532\2067151446.py:9: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  text = BeautifulSoup(text, "html.parser").get_text()


In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    stop_words=None
)

X = vectorizer.fit_transform(df["text_clean"])
y = df["prdtypecode"]


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

preds = model.predict(X_val)

print("F1 score :", f1_score(y_val, preds, average="weighted"))

F1 score : 0.809959415827279


In [6]:
import joblib

joblib.dump(model, "model.joblib")
joblib.dump(vectorizer, "vectorizer.joblib")

['vectorizer.joblib']